## Cell 1: Find the RGI data and check the RGI 6 link.

Locate the repository, inventory its data files, find the RGI 7-to-RGI 6 link files, and count how many glaciers retain their RGI 6 outlines.

#### Notes

- Locate the repository folder containing .git, whether the notebook runs from the root or notebooks/.
- List repository data files while skipping .git.
- Find the RGI 7-to-RGI 6 link files.
- Find the glacier attribute tables.


In [2]:
from pathlib import Path
import pandas as pd


ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists()), Path.cwd())
print('Repo folder:', ROOT)


EXT = {'.csv', '.shp', '.gpkg', '.parquet', '.geojson', '.zip', '.json'}
files = sorted(p for p in ROOT.rglob('*')
               if p.is_file() and p.suffix.lower() in EXT and '.git' not in p.parts)
print(f'\n{len(files)} data files found:')
for p in files[:80]:
    print(f'  {p.relative_to(ROOT)}   {p.stat().st_size / 1e6:.1f} MB')
if len(files) > 80:
    print(f'  ... and {len(files) - 80} more')


links = [p for p in files if 'rgi6_links' in p.name.lower()]
print('\nLink files found:', [str(p.relative_to(ROOT)) for p in links] or 'none')
for p in links:
    df = pd.read_csv(p)
    print('\n', p.name, df.shape)
    print(df.columns.tolist())
    print(df.head(5).to_string())


attr_files = [p for p in files if p.suffix.lower() == '.csv' and 'attributes' in p.name.lower()]
print('\nAttribute files found:', [str(p.relative_to(ROOT)) for p in attr_files] or 'none')
if attr_files:
    attrs = pd.concat([pd.read_csv(p) for p in attr_files])
    print(f'{len(attrs):,} glaciers')
    print(attrs.columns.tolist())
    if 'is_rgi6' in attrs.columns:
        print(attrs['is_rgi6'].value_counts())

Repo folder: d:\nepal-glacier-explorer

30 data files found:
  .vscode\settings.json   0.0 MB
  data\processed\nepal_glaciers.gpkg   16.8 MB
  data\processed\nepal_hypsometry.parquet   0.3 MB
  data\processed\nepal_hypsometry_km2.parquet   0.8 MB
  data\raw\boundary\geoBoundaries-NPL-ADM0-all\geoBoundaries-NPL-ADM0-metaData.json   0.0 MB
  data\raw\boundary\geoBoundaries-NPL-ADM0-all\geoBoundaries-NPL-ADM0.geojson   2.9 MB
  data\raw\boundary\geoBoundaries-NPL-ADM0-all\geoBoundaries-NPL-ADM0.shp   1.1 MB
  data\raw\boundary\geoBoundaries-NPL-ADM0-all\geoBoundaries-NPL-ADM0_simplified.geojson   0.2 MB
  data\raw\boundary\geoBoundaries-NPL-ADM0-all\geoBoundaries-NPL-ADM0_simplified.shp   0.1 MB
  data\raw\rgi7\RGI2000-v7.0-G-14_south_asia_west\RGI2000-v7.0-G-14_south_asia_west-attributes.csv   10.1 MB
  data\raw\rgi7\RGI2000-v7.0-G-14_south_asia_west\RGI2000-v7.0-G-14_south_asia_west-attributes_metadata.json   0.0 MB
  data\raw\rgi7\RGI2000-v7.0-G-14_south_asia_west\RGI2000-v7.0-G-14_sou

## Cell 2: List the required data files.

Log in to NASA Earthdata and list the names and sizes of the HMA2_GGP glacier projections and NSIDC-0770 RGI 6.0 outlines.


In [5]:
import earthaccess

earthaccess.login()

def list_files(short_name, versions):
    results = []
    for v in versions:
        results = earthaccess.search_data(short_name=short_name, version=v, count=-1)
        if results:
            break
    print(f'\n{short_name}: {len(results)} files')
    for r in results:
        for url in r.data_links():
            print('  ', url.split('/')[-1], f'  {r.size():.0f} MB')
    return results

proj_files = list_files('HMA2_GGP', ['001', '1'])
rgi6_files = list_files('NSIDC-0770', ['6', '006'])

d:\miniconda\envs\nepal-glaciers\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
C:\Users\LOQ\AppData\Local\Temp\ipykernel_25900\2084007303.py:27: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  print('  ', url.split('/')[-1], f'  {r.size():.0f} MB')



HMA2_GGP: 131 files
   R06_glac_runoff_fixed_monthly_1set_2000_2100-ssp370-Batch-1-1000.nc   63 MB
   R16_glac_mass_bsl_annual_50sets_2000_2100-rcp26.nc   4 MB
   R16_glac_mass_bsl_annual_50sets_2000_2100-rcp45.nc   4 MB
   R16_glac_mass_bsl_annual_50sets_2000_2100-rcp85.nc   4 MB
   R16_glac_mass_bsl_annual_50sets_2000_2100-ssp119.nc   4 MB
   R16_glac_mass_bsl_annual_50sets_2000_2100-ssp126.nc   4 MB
   R16_glac_mass_bsl_annual_50sets_2000_2100-ssp245.nc   4 MB
   R16_glac_mass_bsl_annual_50sets_2000_2100-ssp370.nc   4 MB
   R16_glac_mass_bsl_annual_50sets_2000_2100-ssp585.nc   4 MB
   R16_glac_mass_bsl_annual_mad_50sets_2000_2100-rcp26.nc   4 MB
   R16_glac_mass_bsl_annual_mad_50sets_2000_2100-rcp45.nc   4 MB
   R16_glac_mass_bsl_annual_mad_50sets_2000_2100-rcp85.nc   4 MB
   R16_glac_mass_bsl_annual_mad_50sets_2000_2100-ssp119.nc   4 MB
   R16_glac_mass_bsl_annual_mad_50sets_2000_2100-ssp126.nc   4 MB
   R16_glac_mass_bsl_annual_mad_50sets_2000_2100-ssp245.nc   4 MB
   R16_glac_ma

## Cell 3: Download glacier projections for SSP245 and SSP585.

Download the yearly glacier-area projections for RGI region 15 into data/raw/projections/. For each scenario, Retrieve the central estimate and spread, then inspect one file.

#### Notes

- Inspect one file without loading the full dataset into memory.


In [7]:
from pathlib import Path
import shutil
import xarray as xr
import earthaccess

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists()), Path.cwd())
DEST = ROOT / 'data' / 'raw' / 'projections'
DEST.mkdir(parents=True, exist_ok=True)

wanted = {f'R15_glac_area_annual{k}_50sets_2000_2100-{s}.nc'
          for k in ('', '_mad') for s in ('ssp245', 'ssp585')}

earthaccess.login()

results = []
for version in ('1', '001'):
    results = earthaccess.search_data(short_name='HMA2_GGP', version=version, count=-1)
    if results:
        print(f'Found {len(results)} files in the data set (version {version})')
        break

chosen = [r for r in results if any(u.split('/')[-1] in wanted for u in r.data_links())]
print(f'{len(chosen)} of {len(wanted)} wanted files found')
if not chosen:
    raise ValueError('The wanted files were not found. Send me the lines above.')

for r in chosen:
    for u in r.data_links():
        print('  ', u.split('/')[-1])

print(f'\nFree disk space: {shutil.disk_usage(DEST).free / 1e9:.0f} GB')
earthaccess.download(chosen, str(DEST))


f = DEST / 'R15_glac_area_annual_50sets_2000_2100-ssp245.nc'
if f.exists():
    print('\n', f.name)
    print(xr.open_dataset(f))

d:\miniconda\envs\nepal-glaciers\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
d:\miniconda\envs\nepal-glaciers\Lib\site-packages\earthaccess\store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


Found 131 files in the data set (version 1)
1 of 4 wanted files found
   R15_glac_area_annual_50sets_2000_2100-rcp26.nc
   R15_glac_area_annual_50sets_2000_2100-rcp45.nc
   R15_glac_area_annual_50sets_2000_2100-rcp85.nc
   R15_glac_area_annual_50sets_2000_2100-ssp119.nc
   R15_glac_area_annual_50sets_2000_2100-ssp126.nc
   R15_glac_area_annual_50sets_2000_2100-ssp245.nc
   R15_glac_area_annual_50sets_2000_2100-ssp585.nc
   R15_glac_area_annual_50sets_2000_2100-ssp370.nc
   R15_glac_area_annual_mad_50sets_2000_2100-rcp26.nc
   R15_glac_area_annual_mad_50sets_2000_2100-rcp45.nc
   R15_glac_area_annual_mad_50sets_2000_2100-rcp85.nc
   R15_glac_area_annual_mad_50sets_2000_2100-ssp119.nc
   R15_glac_area_annual_mad_50sets_2000_2100-ssp126.nc
   R15_glac_area_annual_mad_50sets_2000_2100-ssp245.nc
   R15_glac_area_annual_mad_50sets_2000_2100-ssp370.nc
   R15_glac_area_annual_mad_50sets_2000_2100-ssp585.nc

Free disk space: 171 GB


QUEUEING TASKS | : 100%|██████████| 16/16 [00:00<00:00, 1276.20it/s]
PROCESSING TASKS | : 100%|██████████| 16/16 [00:28<00:00,  1.79s/it]
COLLECTING RESULTS | : 100%|██████████| 16/16 [00:00<?, ?it/s]



 R15_glac_area_annual_50sets_2000_2100-ssp245.nc
<xarray.Dataset> Size: 129MB
Dimensions:           (model: 12, glacier: 13119, year: 102)
Coordinates:
    Climate_Model     (model) <U13 624B ...
    RGIId             (glacier) <U14 735kB ...
    lon               (glacier) float64 105kB ...
    lat               (glacier) float64 105kB ...
  * year              (year) int64 816B 2000 2001 2002 2003 ... 2099 2100 2101
Dimensions without coordinates: model, glacier
Data variables:
    glac_area_annual  (model, glacier, year) float64 128MB ...
    crs               float64 8B ...
Attributes:
    Region:       15 - South Asia East
    source:       PyGEMv0.1.0
    institution:  Carnegie Mellon University
    history:      Created by David Rounce (drounce@cmu.edu) on January 30 2021
    references:   doi:10.3389/feart.2019.00331 and doi:10.1017/jog.2019.91
    Conventions:  CF-1.9
    featureType:  timeSeries


## Cell 4: Make a first-pass match between RGI 7 glaciers and projection glaciers.

Find RGI 6 centre points inside each RGI 7 outline, check whether their sizes agree, report the matched glacier count, and verify the units by summing 2000 areas.

#### Notes

- Load the RGI 7 glaciers.
- Load the RGI 6 projection glacier IDs, centre points, and 2000 areas.
- Find which RGI 6 centre points fall inside each RGI 7 outline.
- Summarize the matches by the website size classes.


In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists()), Path.cwd())
PROJ = ROOT / 'data' / 'raw' / 'projections' / 'R15_glac_area_annual_50sets_2000_2100-ssp245.nc'


g7 = gpd.read_file(ROOT / 'data' / 'processed' / 'nepal_glaciers.gpkg')[['rgi_id', 'area_km2', 'geometry']]
g7 = g7.to_crs(4326)


ds = xr.open_dataset(PROJ)
print('Climate models:', list(ds['Climate_Model'].values))
area2000 = ds['glac_area_annual'].isel(year=0).median('model').values
print(f'Total area of the {len(area2000):,} glaciers in 2000: {np.nansum(area2000) / 1e6:,.0f} km2 '
      '(RGI 6 lists about 14,700 km2 for the region, so this checks the units are m2)')

rgi6 = pd.DataFrame({'rgi6_id': ds['RGIId'].values.astype(str),
                     'lon': ds['lon'].values, 'lat': ds['lat'].values,
                     'area2000_m2': area2000})
pts = gpd.GeoDataFrame(rgi6, geometry=gpd.points_from_xy(rgi6.lon, rgi6.lat), crs=4326)


hit = gpd.sjoin(pts, g7, how='inner', predicate='within').drop_duplicates('rgi6_id')
by7 = hit.groupby('rgi_id').agg(n_rgi6=('rgi6_id', 'count'),
                                rgi6_ids=('rgi6_id', ';'.join),
                                rgi6_area_km2=('area2000_m2', lambda s: s.sum() / 1e6)).reset_index()

res = g7[['rgi_id', 'area_km2']].merge(by7, on='rgi_id', how='left')
res['n_rgi6'] = res['n_rgi6'].fillna(0).astype(int)
res['size_ratio'] = res['rgi6_area_km2'] / res['area_km2']
matched = res['n_rgi6'] > 0
good = matched & res['size_ratio'].between(0.5, 2)

n, total = len(res), res['area_km2'].sum()
print(f'\nRGI 7 glaciers: {n:,}')
print(f'  with a matched RGI 6 glacier : {matched.sum():,} ({matched.mean():.0%}), '
      f'{res.loc[matched, "area_km2"].sum() / total:.0%} of the area')
print(f'  ... and sizes agree (0.5x-2x): {good.sum():,} ({good.mean():.0%}), '
      f'{res.loc[good, "area_km2"].sum() / total:.0%} of the area')
print(f'  with several RGI 6 glaciers  : {(res.n_rgi6 > 1).sum():,}')


res['size_class'] = pd.cut(res['area_km2'], [0, 0.5, 1, 5, 10, 50, np.inf],
                           labels=['<0.5', '0.5-1', '1-5', '5-10', '10-50', '>50'])
summary = res.groupby('size_class', observed=True).agg(
    glaciers=('rgi_id', 'count'), matched=('n_rgi6', lambda s: int((s > 0).sum())),
    area_km2=('area_km2', 'sum'))
summary['matched_%'] = (100 * summary['matched'] / summary['glaciers']).round(0)
print('\n', summary.round(0))

out = ROOT / 'data' / 'processed' / 'rgi7_rgi6_centroid_matches.csv'
res.drop(columns='size_class').to_csv(out, index=False)
print('\nSaved', out)

Climate models: [np.str_('BCC-CSM2-MR'), np.str_('CESM2'), np.str_('CESM2-WACCM'), np.str_('EC-Earth3'), np.str_('EC-Earth3-Veg'), np.str_('FGOALS-f3-L'), np.str_('GFDL-ESM4'), np.str_('INM-CM4-8'), np.str_('INM-CM5-0'), np.str_('MPI-ESM1-2-HR'), np.str_('MRI-ESM2-0'), np.str_('NorESM2-MM')]
Total area of the 13,119 glaciers in 2000: 14,734 km2 (RGI 6 lists about 14,700 km2 for the region, so this checks the units are m2)

RGI 7 glaciers: 4,679
  with a matched RGI 6 glacier : 2,970 (63%), 89% of the area
  ... and sizes agree (0.5x-2x): 2,826 (60%), 85% of the area
  with several RGI 6 glaciers  : 274

             glaciers  matched  area_km2  matched_%
size_class                                        
<0.5            3147     1649     525.0       52.0
0.5-1            634      524     446.0       83.0
1-5              709      621    1485.0       88.0
5-10              90       84     643.0       93.0
10-50             93       86    1749.0       92.0
>50                6        6  

## Cell 5: Build per-glacier projection files.

For each RGI 7 glacier matched in Cell 4, Calculate the share of its 2000 area remaining through 2100 for SSP245 and SSP585. Store the model median and the 10th-to-90th-percentile range in one JSON file per glacier under docs/data/proj/.

#### Notes

- Load yearly areas as model-by-glacier-by-year arrays.
- Use the 2101 slice as the end-of-2100 value.
- Sum matched glacier area for each model and year.
- Create one output file per glacier.
- Run sanity checks on the outputs.


In [9]:
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists()), Path.cwd())
RAW = ROOT / 'data' / 'raw' / 'projections'
OUT = ROOT / 'docs' / 'data' / 'proj'
OUT.mkdir(parents=True, exist_ok=True)

SCENARIOS = ['ssp245', 'ssp585']
SOURCE = ('Rounce et al. 2023, PyGEM-OGGM glacier projections (HMA2_GGP, NSIDC), '
          'median of 12 climate models; matched to RGI 7 by centre point')

matches = pd.read_csv(ROOT / 'data' / 'processed' / 'rgi7_rgi6_centroid_matches.csv')
matches = matches[matches['n_rgi6'] > 0]
print(f'{len(matches):,} glaciers have a match')


areas, ids6, years = {}, None, None
for scn in SCENARIOS:
    ds = xr.open_dataset(RAW / f'R15_glac_area_annual_50sets_2000_2100-{scn}.nc')
    yr = ds['year'].values
    keep = yr <= 2100                       
    areas[scn] = ds['glac_area_annual'].values[:, :, keep]
    these = ds['RGIId'].values.astype(str)
    if ids6 is None:
        ids6, years = these, yr[keep]
    elif not np.array_equal(ids6, these):
        raise ValueError(f'Glacier order is different in {scn}')
    print(scn, 'array shape (models, glaciers, years):', areas[scn].shape)
index6 = {g: i for i, g in enumerate(ids6)}


def clean(a):
    """Round, and turn missing values into null so the file is valid JSON."""
    return [round(float(v), 1) if np.isfinite(v) else None for v in a]


def percent_left(scn, idx):
    """Share of the 2000 area still left, for each model and year."""
    a = areas[scn][:, idx, :].sum(axis=1)          
    with np.errstate(divide='ignore', invalid='ignore'):
        return 100 * a / a[:, :1]



index_out, written = {}, 0
total_area = 0.0
left_area = {s: 0.0 for s in SCENARIOS}
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for row in matches.itertuples():
        idx = [index6[g] for g in row.rgi6_ids.split(';') if g in index6]
        if not idx:
            continue
        rec = {'id': row.rgi_id, 'area_km2': round(float(row.area_km2), 3),
               'n_rgi6': len(idx),
               'size_ratio': round(float(row.size_ratio), 2),
               'years': [int(years[0]), int(years[-1])],
               'source': SOURCE}
        end_values = []
        for scn in SCENARIOS:
            pct = percent_left(scn, idx)
            med = np.nanmedian(pct, axis=0)
            p10, p90 = np.nanpercentile(pct, [10, 90], axis=0)
            rec[scn] = {'median': clean(med), 'p10': clean(p10), 'p90': clean(p90)}
            end_values.append(round(float(med[-1]), 1))
            if np.isfinite(med[-1]):
                left_area[scn] += row.area_km2 * med[-1] / 100
        total_area += row.area_km2
        (OUT / f'{row.rgi_id}.json').write_text(json.dumps(rec, separators=(',', ':')))
        index_out[row.rgi_id] = end_values
        written += 1

(OUT / 'index.json').write_text(json.dumps(index_out, separators=(',', ':')))
size_mb = sum(p.stat().st_size for p in OUT.glob('*.json')) / 1e6
print(f'\nWrote {written:,} glacier files ({size_mb:.1f} MB) to {OUT}')


print('\nAll matched glaciers together (area-weighted), share of 2000 area left in 2100:')
for scn in SCENARIOS:
    print(f'  {scn}: {100 * left_area[scn] / total_area:.0f}%')

ex = 'RGI2000-v7.0-G-15-05847'
f = OUT / f'{ex}.json'
if f.exists():
    r = json.loads(f.read_text())
    print(f"\nExample {ex}: {r['area_km2']} km2, matched {r['n_rgi6']} projection glacier(s)")
    for scn in SCENARIOS:
        m, lo, hi = r[scn]['median'], r[scn]['p10'], r[scn]['p90']
        print(f'  {scn}: 2050 {m[50]}%  (band {lo[50]}-{hi[50]})   2100 {m[100]}%  (band {lo[100]}-{hi[100]})')
else:
    print(f'\n{ex} was not matched (no file written for it).')

2,970 glaciers have a match
ssp245 array shape (models, glaciers, years): (12, 13119, 101)
ssp585 array shape (models, glaciers, years): (12, 13119, 101)

Wrote 2,970 glacier files (9.5 MB) to d:\nepal-glacier-explorer\docs\data\proj

All matched glaciers together (area-weighted), share of 2000 area left in 2100:
  ssp245: 30%
  ssp585: 15%

Example RGI2000-v7.0-G-15-05847: 11.799 km2, matched 1 projection glacier(s)
  ssp245: 2050 91.7%  (band 89.9-94.1)   2100 38.7%  (band 31.6-54.7)
  ssp585: 2050 91.7%  (band 90.7-92.0)   2100 19.8%  (band 15.3-33.5)


## Cell 6: Download RGI 6 outlines and region 14 projections.

Download and unpack the RGI 6.0 outlines for regions 14 and 15 from glims.org, download region 14 projection files from NASA, and check glacier counts and total area.

#### Notes

- Download and unpack the RGI 6.0 outlines.
- Download the region 14 projections.


In [10]:
import shutil
import urllib.request
import zipfile
from pathlib import Path
import geopandas as gpd
import earthaccess

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists()), Path.cwd())
RGI6_DIR = ROOT / 'data' / 'raw' / 'rgi6'
PROJ_DIR = ROOT / 'data' / 'raw' / 'projections'
RGI6_DIR.mkdir(parents=True, exist_ok=True)
PROJ_DIR.mkdir(parents=True, exist_ok=True)


OUTLINES = {'14': '14_rgi60_SouthAsiaWest', '15': '15_rgi60_SouthAsiaEast'}
BASES = ['https://www.glims.org/RGI/rgi60_files/', 'http://www.glims.org/RGI/rgi60_files/']

def download(url, dest):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r, open(dest, 'wb') as f:
        shutil.copyfileobj(r, f)

for reg, stem in OUTLINES.items():
    zip_path = RGI6_DIR / f'{stem}.zip'
    folder = RGI6_DIR / stem
    if not (zip_path.exists() and zip_path.stat().st_size > 1e6):
        for base in BASES:
            try:
                print('downloading', base + stem + '.zip')
                download(base + stem + '.zip', zip_path)
                break
            except Exception as e:
                print('   failed:', e)
    if zip_path.exists() and zip_path.stat().st_size > 1e6:
        if not folder.exists():
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(folder)
        shp = next(folder.rglob('*.shp'), None)
        if shp is not None:
            g = gpd.read_file(shp)
            print(f'Region {reg}: {len(g):,} glaciers, total area {g["Area"].sum():,.0f} km2   ({shp.name})')
    else:
        print(f'Region {reg}: could not download. Get {stem}.zip from the RGI 6.0 page at '
              f'glims.org/RGI and put it in {RGI6_DIR}')


wanted = {f'R14_glac_area_annual{k}_50sets_2000_2100-{s}.nc'
          for k in ('', '_mad') for s in ('ssp245', 'ssp585')}
earthaccess.login()
results = []
for version in ('1', '001'):
    results = earthaccess.search_data(short_name='HMA2_GGP', version=version, count=-1)
    if results:
        break
chosen = [r for r in results if any(u.split('/')[-1] in wanted for u in r.data_links())]
print(f'\nRegion 14 projection files: {len(chosen)} package(s) found')
if chosen:
    earthaccess.download(chosen, str(PROJ_DIR))
print('Files now in', PROJ_DIR)
for p in sorted(PROJ_DIR.glob('R14_glac_area_annual_50sets_*ssp*.nc')):
    print('  ', p.name)

downloading https://www.glims.org/RGI/rgi60_files/14_rgi60_SouthAsiaWest.zip
Region 14: could not download. Get 14_rgi60_SouthAsiaWest.zip from the RGI 6.0 page at glims.org/RGI and put it in d:\nepal-glacier-explorer\data\raw\rgi6
downloading https://www.glims.org/RGI/rgi60_files/15_rgi60_SouthAsiaEast.zip
Region 15: could not download. Get 15_rgi60_SouthAsiaEast.zip from the RGI 6.0 page at glims.org/RGI and put it in d:\nepal-glacier-explorer\data\raw\rgi6


d:\miniconda\envs\nepal-glaciers\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
d:\miniconda\envs\nepal-glaciers\Lib\site-packages\earthaccess\store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)



Region 14 projection files: 1 package(s) found


QUEUEING TASKS | : 100%|██████████| 16/16 [00:00<00:00, 1447.25it/s]
PROCESSING TASKS | : 100%|██████████| 16/16 [00:57<00:00,  3.60s/it]
COLLECTING RESULTS | : 100%|██████████| 16/16 [00:00<?, ?it/s]

Files now in d:\nepal-glacier-explorer\data\raw\projections
   R14_glac_area_annual_50sets_2000_2100-ssp119.nc
   R14_glac_area_annual_50sets_2000_2100-ssp126.nc
   R14_glac_area_annual_50sets_2000_2100-ssp245.nc
   R14_glac_area_annual_50sets_2000_2100-ssp370.nc
   R14_glac_area_annual_50sets_2000_2100-ssp585.nc


## Cell 7: Load and check the RGI 6 outlines.

Locate the RGI 6.0 archives in data/raw/rgi6/, unpack them, and report the glacier count and total area for each region.

#### Notes

- Remove tiny invalid ZIP files left by failed downloads that saved an error page.


In [12]:
import zipfile
from pathlib import Path
import geopandas as gpd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists()), Path.cwd())
RGI6_DIR = ROOT / 'data' / 'raw' / 'rgi6'


for p in RGI6_DIR.glob('*.zip'):
    if p.stat().st_size < 1_000_000:
        print('removing tiny file', p.name, f'({p.stat().st_size} bytes)')
        p.unlink()

for reg, key in {'14': 'SouthAsiaWest', '15': 'SouthAsiaEast'}.items():
    zips = [p for p in RGI6_DIR.glob('*.zip') if key.lower() in p.name.lower()]
    if not zips:
        print(f'Region {reg}: no zip with "{key}" in its name found in {RGI6_DIR}')
        continue
    folder = RGI6_DIR / f'rgi6_region_{reg}'
    if not folder.exists():
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(folder)
    shp = next(folder.rglob('*.shp'), None)
    g = gpd.read_file(shp)
    print(f'Region {reg} ({zips[0].name}): {len(g):,} glaciers, {g["Area"].sum():,.0f} km2')
    print('   columns:', list(g.columns))

Region 14 (14_rgi60_SouthAsiaWest.zip): 27,988 glaciers, 33,568 km2
   columns: ['RGIId', 'GLIMSId', 'BgnDate', 'EndDate', 'CenLon', 'CenLat', 'O1Region', 'O2Region', 'Area', 'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'Status', 'Connect', 'Form', 'TermType', 'Surging', 'Linkages', 'Name', 'geometry']
Region 15 (15_rgi60_SouthAsiaEast.zip): 13,119 glaciers, 14,734 km2
   columns: ['RGIId', 'GLIMSId', 'BgnDate', 'EndDate', 'CenLon', 'CenLat', 'O1Region', 'O2Region', 'Area', 'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'Status', 'Connect', 'Form', 'TermType', 'Surging', 'Linkages', 'Name', 'geometry']


## Cell 8: Match the glacier outlines to projection glaciers by overlap.

Measure the overlap between the RGI 7 glacier outlines and the RGI 6 outlines in regions 14 and 15. Retain old glaciers covering at least 10% of each RGI 7 glacier, or the largest overlap when none meet that threshold. Classify coverage as good (at least 60%), partial (30 to 60%), weak (under 30%), or none (no overlap), then save the matches for Cell 9.

#### Notes

- Use an equal-area projection so area measurements are accurate.
- Require an old glacier to cover at least 10% of the new glacier.
- Load the RGI 7 glaciers.
- Load nearby RGI 6 outlines from regions 14 and 15.
- Calculate the glacier-outline overlaps.
- Calculate the share of the new glacier covered by each old glacier.
- Calculate the share of each old glacier inside the new glacier.
- Report the overlap matches.


In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists()), Path.cwd())
RGI6_DIR = ROOT / 'data' / 'raw' / 'rgi6'
CRS_AREA = 6933            
MIN_FRAC7 = 0.10           


g7 = gpd.read_file(ROOT / 'data' / 'processed' / 'nepal_glaciers.gpkg')[['rgi_id', 'area_km2', 'geometry']]
g7 = g7.to_crs(CRS_AREA)
g7['geometry'] = g7.geometry.make_valid()
g7['area7_km2'] = g7.geometry.area / 1e6


parts = []
for reg in ('14', '15'):
    shp = next((RGI6_DIR / f'rgi6_region_{reg}').rglob('*.shp'))
    parts.append(gpd.read_file(shp)[['RGIId', 'Area', 'geometry']].to_crs(CRS_AREA))
g6 = pd.concat(parts, ignore_index=True).rename(columns={'Area': 'area6_km2'})
g6 = gpd.GeoDataFrame(g6, geometry='geometry', crs=CRS_AREA)
g6['geometry'] = g6.geometry.make_valid()
minx, miny, maxx, maxy = g7.total_bounds
g6 = g6.cx[minx - 5000:maxx + 5000, miny - 5000:maxy + 5000]
print(f'{len(g7):,} RGI 7 glaciers, {len(g6):,} RGI 6 glaciers nearby')


print('measuring overlaps ...')
pieces = gpd.overlay(g7[['rgi_id', 'area7_km2', 'geometry']],
                     g6[['RGIId', 'area6_km2', 'geometry']],
                     how='intersection', keep_geom_type=True)
pieces['overlap_km2'] = pieces.geometry.area / 1e6
pieces = pieces[pieces['overlap_km2'] > 0].copy()
pieces['frac7'] = pieces['overlap_km2'] / pieces['area7_km2']    
pieces['frac6'] = pieces['overlap_km2'] / pieces['area6_km2']    

kept = pieces[pieces['frac7'] >= MIN_FRAC7]
best = pieces.sort_values('overlap_km2', ascending=False).drop_duplicates('rgi_id')
kept = pd.concat([kept, best[~best['rgi_id'].isin(kept['rgi_id'])]])

by7 = kept.groupby('rgi_id').agg(
    n_rgi6=('RGIId', 'count'),
    rgi6_ids=('RGIId', ';'.join),
    weights=('frac6', lambda s: ';'.join(f'{w:.4f}' for w in s)),
    coverage=('frac7', 'sum')).reset_index()
by7['coverage'] = by7['coverage'].clip(upper=1.0)

res = g7[['rgi_id', 'area_km2']].merge(by7, on='rgi_id', how='left')
res['n_rgi6'] = res['n_rgi6'].fillna(0).astype(int)
res['coverage'] = res['coverage'].fillna(0.0)
res['quality'] = np.select([res.coverage >= 0.6, res.coverage >= 0.3, res.coverage > 0],
                           ['good', 'partial', 'weak'], default='none')


n, total = len(res), res['area_km2'].sum()
print(f'\nRGI 7 glaciers: {n:,}')
for q in ['good', 'partial', 'weak', 'none']:
    sel = res['quality'] == q
    print(f'  {q:8s}: {sel.sum():5,} glaciers ({sel.mean():.0%}), '
          f'{res.loc[sel, "area_km2"].sum() / total:.0%} of the area')

res['size_class'] = pd.cut(res['area_km2'], [0, 0.5, 1, 5, 10, 50, np.inf],
                           labels=['<0.5', '0.5-1', '1-5', '5-10', '10-50', '>50'])
table = res.pivot_table(index='size_class', columns='quality', values='rgi_id',
                        aggfunc='count', observed=True, fill_value=0)
print('\nGlaciers per size class (km2) and match quality:\n', table)

out = ROOT / 'data' / 'processed' / 'rgi7_rgi6_overlap_matches.csv'
res.drop(columns='size_class').to_csv(out, index=False)
print('\nSaved', out)

4,679 RGI 7 glaciers, 5,502 RGI 6 glaciers nearby
measuring overlaps ...

RGI 7 glaciers: 4,679
  good    : 2,882 glaciers (62%), 84% of the area
  partial :   277 glaciers (6%), 9% of the area
  weak    :   181 glaciers (4%), 4% of the area
  none    : 1,339 glaciers (29%), 4% of the area

Glaciers per size class (km2) and match quality:
 quality     good  none  partial  weak
size_class                           
<0.5        1654  1258      127   108
0.5-1        485    62       59    28
1-5          581    19       71    38
5-10          75     0       12     3
10-50         82     0        7     4
>50            5     0        1     0

Saved d:\nepal-glacier-explorer\data\processed\rgi7_rgi6_overlap_matches.csv


## Cell 9: Build per-glacier projection files for all scenarios.

For each RGI 7 glacier matched in Cell 8, Calculate the share of its 2000 area remaining through 2100 for all downloaded SSP and RCP scenarios. Store the model median and the 10th-to-90th-percentile range, using both regions, in docs/data/proj/.

#### Notes

- Start with a clean output folder.
- Map an RGI 6 ID such as RGI60-15.00001 to its region label R15.
- Assign each border glacier to the region containing most of its matched area.
- Load only the old glaciers required for the matches.
- Create one output file per glacier.


In [16]:
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists()), Path.cwd())
RAW = ROOT / 'data' / 'raw' / 'projections'
OUT = ROOT / 'docs' / 'data' / 'proj'
OUT.mkdir(parents=True, exist_ok=True)
for p in OUT.glob('*.json'):          
    p.unlink()

SCENARIOS = ['ssp119', 'ssp126', 'ssp245', 'ssp370', 'ssp585', 'rcp26', 'rcp45', 'rcp85']
REGIONS = ['R14', 'R15']
SOURCE = ('Rounce et al. 2023, PyGEM-OGGM glacier projections (HMA2_GGP, NSIDC): median of the '
          'climate models; RGI 7 glaciers matched to the RGI 6 glaciers they overlap')

matches = pd.read_csv(ROOT / 'data' / 'processed' / 'rgi7_rgi6_overlap_matches.csv')
matches = matches[matches['n_rgi6'] > 0].copy()
matches['ids'] = matches['rgi6_ids'].str.split(';')
matches['w'] = matches['weights'].astype(str).str.split(';').apply(lambda L: [float(x) for x in L])
print(f'{len(matches):,} glaciers have a match')


def region_of(rgi6_id):               
    return 'R' + rgi6_id.split('-')[1].split('.')[0]



plan, need = {}, {r: set() for r in REGIONS}
for row in matches.itertuples():
    groups = {}
    for g, x in zip(row.ids, row.w):
        groups.setdefault(region_of(g), []).append((g, x))
    reg = max(groups, key=lambda r: sum(x for _, x in groups[r]))
    if reg not in need:
        continue
    plan[row.rgi_id] = (reg, [g for g, _ in groups[reg]], np.array([x for _, x in groups[reg]]))
    need[reg].update(g for g, _ in groups[reg])


data = {}
for reg in REGIONS:
    for scn in SCENARIOS:
        f = RAW / f'{reg}_glac_area_annual_50sets_2000_2100-{scn}.nc'
        if not f.exists():
            print('missing file, skipped:', f.name)
            continue
        ds = xr.open_dataset(f)
        all_ids = ds['RGIId'].values.astype(str)
        pos = {g: i for i, g in enumerate(all_ids)}
        chosen = [g for g in sorted(need[reg]) if g in pos]
        block = (ds['glac_area_annual'].isel(glacier=[pos[g] for g in chosen])
                 .sel(year=slice(2000, 2100)).values)
        data[(reg, scn)] = (block, {g: k for k, g in enumerate(chosen)})
        print(f'{reg} {scn}: {ds.sizes["model"]} models, {len(chosen):,} glaciers loaded')


def clean(a):
    """Round (one decimal below 10%, whole numbers above) and turn missing values into null."""
    out = []
    for v in a:
        if not np.isfinite(v):
            out.append(None)
        elif v < 10:
            out.append(round(float(v), 1))
        else:
            out.append(int(round(float(v))))
    return out



index_out, counts, skipped = {}, {}, 0
left_area = {s: 0.0 for s in SCENARIOS}
area_for = {s: 0.0 for s in SCENARIOS}
rows = matches.set_index('rgi_id')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for rgi_id, (reg, ids, w) in plan.items():
        row = rows.loc[rgi_id]
        area = float(row['area_km2'])
        rec = {'id': rgi_id, 'area_km2': round(area, 3),
               'match': {'quality': row['quality'], 'coverage': round(float(row['coverage']), 2),
                         'n_rgi6': len(ids)},
               'years': [2000, 2100], 'source': SOURCE}
        end_values = {}
        for scn in SCENARIOS:
            if (reg, scn) not in data:
                continue
            block, pos = data[(reg, scn)]
            keep = [i for i, g in enumerate(ids) if g in pos]
            if not keep:
                continue
            a = np.einsum('mgy,g->my', block[:, [pos[ids[i]] for i in keep], :], w[keep])
            with np.errstate(divide='ignore', invalid='ignore'):
                pct = 100 * a / a[:, :1]
            med = np.nanmedian(pct, axis=0)
            p10, p90 = np.nanpercentile(pct, [10, 90], axis=0)
            rec[scn] = {'n': int(block.shape[0]), 'median': clean(med), 'p10': clean(p10), 'p90': clean(p90)}
            end_values[scn] = round(float(med[-1]), 1)
            if np.isfinite(med[-1]):
                left_area[scn] += area * med[-1] / 100
                area_for[scn] += area
        if not end_values:
            skipped += 1
            continue
        (OUT / f'{rgi_id}.json').write_text(json.dumps(rec, separators=(',', ':')))
        index_out[rgi_id] = end_values
        counts[row['quality']] = counts.get(row['quality'], 0) + 1

(OUT / 'index.json').write_text(json.dumps(index_out, separators=(',', ':')))
size_mb = sum(p.stat().st_size for p in OUT.glob('*.json')) / 1e6
print(f'\nWrote {len(index_out):,} glacier files ({size_mb:.1f} MB); skipped {skipped}')
print('By match quality:', counts)

print('\nAll these glaciers together (area-weighted), share of 2000 area left in 2100:')
for scn in SCENARIOS:
    if area_for[scn] > 0:
        print(f'  {scn}: {100 * left_area[scn] / area_for[scn]:.0f}%')

ex = 'RGI2000-v7.0-G-15-05847'
f = OUT / f'{ex}.json'
if f.exists():
    r = json.loads(f.read_text())
    print(f"\nExample {ex}: {r['area_km2']} km2, match {r['match']}")
    for scn in SCENARIOS:
        if scn in r:
            m = r[scn]['median']
            print(f"  {scn} ({r[scn]['n']} models): 2050 {m[50]}%   2100 {m[100]}%")

3,340 glaciers have a match
R14 ssp119: 12 models, 22 glaciers loaded
R14 ssp126: 12 models, 22 glaciers loaded
R14 ssp245: 12 models, 22 glaciers loaded
R14 ssp370: 12 models, 22 glaciers loaded
R14 ssp585: 12 models, 22 glaciers loaded
R14 rcp26: 10 models, 22 glaciers loaded
R14 rcp45: 10 models, 22 glaciers loaded
R14 rcp85: 10 models, 22 glaciers loaded
R15 ssp119: 12 models, 3,236 glaciers loaded
R15 ssp126: 12 models, 3,236 glaciers loaded
R15 ssp245: 12 models, 3,236 glaciers loaded
R15 ssp370: 12 models, 3,236 glaciers loaded
R15 ssp585: 12 models, 3,236 glaciers loaded
R15 rcp26: 10 models, 3,236 glaciers loaded
R15 rcp45: 10 models, 3,236 glaciers loaded
R15 rcp85: 10 models, 3,236 glaciers loaded

Wrote 3,340 glacier files (30.8 MB); skipped 0
By match quality: {'good': 2882, 'weak': 181, 'partial': 277}

All these glaciers together (area-weighted), share of 2000 area left in 2100:
  ssp119: 47%
  ssp126: 43%
  ssp245: 31%
  ssp370: 21%
  ssp585: 16%
  rcp26: 46%
  rcp45: 3

## Cell 10: Add the projection panel to the website.

Generate docs/projection_panel.js and docs/projection_panel.css and link them from docs/index.html. The panel shows projected glacier area from 2000 to 2100, includes controls for toggling scenarios and model ranges, and can expand to nearly full width.


In [ ]:

from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists()), Path.cwd())
DOCS = ROOT / 'docs'

JS = r'''(function () {
  "use strict";

  var ID_RE = /RGI2000-v7\.0-G-\d{2}-\d{5}/;
  var DATA_DIR = "data/proj/";
  var ALL = ["ssp119", "ssp126", "ssp245", "ssp370", "ssp585", "rcp26", "rcp45", "rcp85"];
  var SSPS = ALL.slice(0, 5), RCPS = ALL.slice(5);
  var COLORS = { ssp119: "#00adcf", ssp126: "#173c66", ssp245: "#f79420", ssp370: "#e71d25", ssp585: "#951b1e",
                 rcp26: "#5b8db8", rcp45: "#b8942a", rcp85: "#b0413e" };
  var LABELS = { ssp119: "SSP1-1.9 (very low emissions)", ssp126: "SSP1-2.6 (low)",
                 ssp245: "SSP2-4.5 (middle of the road)", ssp370: "SSP3-7.0 (high)",
                 ssp585: "SSP5-8.5 (very high)", rcp26: "RCP2.6", rcp45: "RCP4.5", rcp85: "RCP8.5" };
  var SHORT = { ssp119: "SSP1-1.9", ssp126: "SSP1-2.6", ssp245: "SSP2-4.5", ssp370: "SSP3-7.0",
                ssp585: "SSP5-8.5", rcp26: "RCP2.6", rcp45: "RCP4.5", rcp85: "RCP8.5" };
  var NS = "http://www.w3.org/2000/svg";

  var checked = { ssp119: true, ssp126: true, ssp245: true, ssp370: true, ssp585: true,
                  rcp26: false, rcp45: false, rcp85: false };
  var showBand = false;
  var currentRec = null, chartHolder = null;
  var selYear = null, markerApi = null, viewer = null;
  var viewerScen = "ssp245", viewerMode = "retreat", viewerRange = "median", viewerSpeed = 10;

  function el(tag, cls, text) {
    var e = document.createElement(tag);
    if (cls) e.className = cls;
    if (text !== undefined) e.textContent = text;
    return e;
  }
  function svg(tag, attrs, parent) {
    var e = document.createElementNS(NS, tag);
    for (var k in attrs) e.setAttribute(k, attrs[k]);
    if (parent) parent.appendChild(e);
    return e;
  }
  function pct(v) { return v === null || v === undefined ? "n/a" : Math.round(v) + "%"; }
  function km2(v) { return v >= 10 ? v.toFixed(1) : v.toFixed(2); }
  function isRcp(sc) { return sc.indexOf("rcp") === 0; }

  /* ---------------------------------------------------------------
     Chart panel (right side)
  --------------------------------------------------------------- */
  var panel = el("div", "proj-panel");
  panel.style.display = "none";
  var closeBtn = el("button", "proj-btn proj-close", "\u00d7");
  closeBtn.title = "Close";
  closeBtn.onclick = function () { panel.style.display = "none"; };
  var wideBtn = el("button", "proj-btn proj-expand", "\u2922");
  wideBtn.title = "Make the panel larger or smaller";
  wideBtn.onclick = function () {
    panel.classList.toggle("proj-wide");
    setTimeout(redraw, 60);
  };
  var heading = el("div", "proj-heading", "Projected area change, 2000 to 2100");
  var idLine = el("div", "proj-id");
  var body = el("div", "proj-body");
  panel.appendChild(wideBtn);
  panel.appendChild(closeBtn);
  panel.appendChild(heading);
  panel.appendChild(idLine);
  panel.appendChild(body);
  document.body.appendChild(panel);

  function chart(rec) {
    markerApi = null;
    var W = Math.max(320, Math.min(1000, (chartHolder && chartHolder.clientWidth) || 560));
    var H = Math.max(250, Math.min(520, Math.round(W * 0.6)));
    var pl = 46, pr = 14, pt = 12, pb = 32;
    var y0 = rec.years[0], y1 = rec.years[1];
    var active = ALL.filter(function (sc) { return rec[sc] && checked[sc]; });
    var ymax = 100;
    active.forEach(function (sc) {
      rec[sc].p90.forEach(function (v) { if (v !== null && v > ymax) ymax = v; });
    });
    ymax = Math.ceil(ymax / 10) * 10;
    var step = ymax <= 100 ? 25 : 20;
    function X(y) { return pl + (y - y0) / (y1 - y0) * (W - pl - pr); }
    function Y(v) { return pt + (1 - v / ymax) * (H - pt - pb); }

    var wrap = el("div", "proj-chartwrap");
    var s = svg("svg", { viewBox: "0 0 " + W + " " + H, width: W, height: H, role: "img" }, wrap);
    s.setAttribute("aria-label", "Projected glacier area as a percent of the 2000 area, 2000 to 2100");

    for (var t = 0; t <= ymax; t += step) {
      svg("line", { x1: pl, x2: W - pr, y1: Y(t), y2: Y(t), stroke: "#e3e3e3", "stroke-width": 1 }, s);
      var lab = svg("text", { x: pl - 6, y: Y(t) + 4, "text-anchor": "end", "font-size": 11, fill: "#666" }, s);
      lab.textContent = t + "%";
    }
    for (var yr = y0; yr <= y1; yr += 25) {
      svg("line", { x1: X(yr), x2: X(yr), y1: H - pb, y2: H - pb + 4, stroke: "#999", "stroke-width": 1 }, s);
      var xl = svg("text", { x: X(yr), y: H - pb + 17, "text-anchor": "middle", "font-size": 11, fill: "#666" }, s);
      xl.textContent = yr;
    }
    var yt = svg("text", { transform: "translate(12," + (pt + (H - pt - pb) / 2) + ") rotate(-90)",
                           "text-anchor": "middle", "font-size": 11, fill: "#666" }, s);
    yt.textContent = "% of 2000 area";

    if (!active.length) {
      var none = svg("text", { x: W / 2, y: H / 2, "text-anchor": "middle", "font-size": 13, fill: "#888" }, s);
      none.textContent = "Tick a scenario below to draw it";
      return wrap;
    }

    if (showBand) {
      active.forEach(function (sc) {
        var p10 = rec[sc].p10, p90 = rec[sc].p90, up = [], down = [];
        for (var i = 0; i < p10.length; i++) {
          if (p10[i] === null || p90[i] === null) continue;
          up.push(X(y0 + i).toFixed(1) + "," + Y(p90[i]).toFixed(1));
          down.unshift(X(y0 + i).toFixed(1) + "," + Y(p10[i]).toFixed(1));
        }
        if (up.length) svg("polygon", { points: up.concat(down).join(" "), fill: COLORS[sc],
                                        "fill-opacity": 0.16, stroke: "none" }, s);
      });
    }
    active.forEach(function (sc) {
      var m = rec[sc].median, pts = [];
      for (var i = 0; i < m.length; i++) {
        if (m[i] !== null) pts.push(X(y0 + i).toFixed(1) + "," + Y(m[i]).toFixed(1));
      }
      var attrs = { points: pts.join(" "), fill: "none", stroke: COLORS[sc],
                    "stroke-width": 2.4, "stroke-linejoin": "round" };
      if (isRcp(sc)) attrs["stroke-dasharray"] = "6 4";
      if (pts.length) svg("polyline", attrs, s);
    });

    /* marker that follows the year slider of the glacier view */
    var mline = svg("line", { y1: pt, y2: H - pb, stroke: "#333", "stroke-width": 1.2, visibility: "hidden" }, s);
    var mdots = {};
    active.forEach(function (sc) {
      mdots[sc] = svg("circle", { r: 4.5, fill: COLORS[sc], stroke: "#fff", "stroke-width": 1.5,
                                  visibility: "hidden" }, s);
    });
    function setMarker(year) {
      if (year === null || year === undefined) {
        mline.setAttribute("visibility", "hidden");
        active.forEach(function (sc) { mdots[sc].setAttribute("visibility", "hidden"); });
        return;
      }
      var i = Math.max(0, Math.min(y1 - y0, Math.round(year - y0)));
      mline.setAttribute("x1", X(y0 + i));
      mline.setAttribute("x2", X(y0 + i));
      mline.setAttribute("visibility", "visible");
      active.forEach(function (sc) {
        var v = rec[sc].median[i];
        if (v === null) { mdots[sc].setAttribute("visibility", "hidden"); return; }
        mdots[sc].setAttribute("cx", X(y0 + i));
        mdots[sc].setAttribute("cy", Y(v));
        mdots[sc].setAttribute("visibility", "visible");
      });
    }
    markerApi = setMarker;
    setMarker(selYear);

    var vline = svg("line", { y1: pt, y2: H - pb, stroke: "#888", "stroke-width": 1,
                              "stroke-dasharray": "3 3", visibility: "hidden" }, s);
    var tip = el("div", "proj-tip");
    tip.style.display = "none";
    wrap.appendChild(tip);
    var hit = svg("rect", { x: pl, y: pt, width: W - pl - pr, height: H - pt - pb, fill: "transparent" }, s);
    function yearAt(ev) {
      var r = s.getBoundingClientRect();
      var px = (ev.clientX - r.left) / r.width * W;
      var yy = Math.round(y0 + (px - pl) / (W - pl - pr) * (y1 - y0));
      return Math.max(y0, Math.min(y1, yy));
    }
    hit.addEventListener("mousemove", function (ev) {
      var yy = yearAt(ev), i = yy - y0;
      vline.setAttribute("x1", X(yy));
      vline.setAttribute("x2", X(yy));
      vline.setAttribute("visibility", "visible");
      tip.textContent = "";
      tip.appendChild(el("div", "proj-tip-year", String(yy)));
      active.forEach(function (sc) {
        var d = el("div", "", SHORT[sc] + ": " + pct(rec[sc].median[i]) + " (" +
                   pct(rec[sc].p10[i]) + " to " + pct(rec[sc].p90[i]) + ")");
        d.style.color = COLORS[sc];
        tip.appendChild(d);
      });
      tip.style.display = "block";
    });
    hit.addEventListener("mouseleave", function () {
      vline.setAttribute("visibility", "hidden");
      tip.style.display = "none";
    });
    hit.addEventListener("click", function (ev) {
      if (viewer) viewer.setYear(yearAt(ev));
    });
    return wrap;
  }

  function redraw() {
    if (!currentRec || !chartHolder) return;
    chartHolder.textContent = "";
    chartHolder.appendChild(chart(currentRec));
  }
  window.addEventListener("resize", redraw);

  function scenarioRow(rec, sc) {
    var d = rec[sc];
    var row = el("label", "proj-row");
    var cb = document.createElement("input");
    cb.type = "checkbox";
    cb.checked = !!checked[sc];
    cb.addEventListener("change", function () { checked[sc] = cb.checked; redraw(); });
    var sw = el("span", "proj-swatch" + (isRcp(sc) ? " proj-dashed" : ""));
    sw.style.borderTopColor = COLORS[sc];
    var m = d.median, last = m.length - 1, end = m[last];
    var txt = el("span", "proj-rowtext");
    txt.appendChild(el("strong", "", LABELS[sc]));
    var detail;
    if (end === null) {
      detail = ": no result";
    } else {
      detail = ": " + pct(end) + " of the 2000 area left in " + rec.years[1] +
               " (about " + km2(rec.area_km2 * end / 100) + " km2; model range " +
               pct(d.p10[last]) + " to " + pct(d.p90[last]) + ")";
      for (var i = 0; i < m.length; i++) {
        if (m[i] !== null && m[i] < 0.5) { detail += ". Gone by about " + (rec.years[0] + i); break; }
      }
    }
    txt.appendChild(document.createTextNode(detail));
    row.appendChild(cb);
    row.appendChild(sw);
    row.appendChild(txt);
    return row;
  }

  function modelsText(rec, list) {
    var seen = [];
    list.forEach(function (sc) { if (rec[sc] && seen.indexOf(rec[sc].n) < 0) seen.push(rec[sc].n); });
    return seen.join(" or ");
  }

  function render(id, rec) {
    body.textContent = "";
    chartHolder = null;
    currentRec = null;
    markerApi = null;
    idLine.textContent = id;
    if (!rec) {
      body.appendChild(el("p", "proj-warn",
        "No projection available for this glacier. The projections were made for the older RGI 6.0 " +
        "inventory, and this glacier does not overlap any glacier in it. It may be a glacier that was " +
        "only mapped in RGI 7.0."));
      return;
    }
    currentRec = rec;

    var bandLabel = el("label", "proj-toggle");
    var bandCb = document.createElement("input");
    bandCb.type = "checkbox";
    bandCb.checked = showBand;
    bandCb.addEventListener("change", function () { showBand = bandCb.checked; redraw(); });
    bandLabel.appendChild(bandCb);
    bandLabel.appendChild(document.createTextNode(" Show model range (10th to 90th percentile)"));
    body.appendChild(bandLabel);

    chartHolder = el("div", "proj-chart");
    body.appendChild(chartHolder);
    redraw();

    var list = el("div", "proj-list");
    SSPS.forEach(function (sc) { if (rec[sc]) list.appendChild(scenarioRow(rec, sc)); });
    var rc = RCPS.filter(function (sc) { return rec[sc]; });
    if (rc.length) {
      var det = el("details", "proj-details");
      det.appendChild(el("summary", "", "Older RCP scenarios (CMIP5 models)"));
      rc.forEach(function (sc) { det.appendChild(scenarioRow(rec, sc)); });
      list.appendChild(det);
    }
    body.appendChild(list);

    var q = rec.match || {};
    if (q.quality && q.quality !== "good") {
      body.appendChild(el("p", "proj-warn",
        (q.quality === "partial" ? "Partial match: " : "Weak match: ") +
        "the old glaciers behind this projection cover only about " + Math.round(q.coverage * 100) +
        "% of this outline. Treat the curves as indicative."));
    }
    var nS = modelsText(rec, SSPS), nR = modelsText(rec, RCPS);
    body.appendChild(el("p", "proj-note",
      "Lines: median of " + (nS || "the") + " climate models for the SSP scenarios" +
      (nR ? " (" + nR + " for the older RCP scenarios, which come from a different model generation)" : "") +
      ". Band: 10th to 90th percentile of the models. Values are the share of the 2000 area left, " +
      "taken from the old RGI 6.0 glacier(s) that overlap this outline; km2 applies that share to the " +
      "RGI 7.0 area. Projections for single small glaciers are much less certain than the regional " +
      "picture. Source: Rounce et al. (2023), PyGEM-OGGM glacier projections, NSIDC."));
  }

  var cache = {}, currentId = null;
  function getRec(id) {
    if (cache[id] !== undefined) return Promise.resolve(cache[id]);
    return fetch(DATA_DIR + id + ".json")
      .then(function (r) { return r.ok ? r.json() : null; })
      .catch(function () { return null; })
      .then(function (rec) { cache[id] = rec; return rec; });
  }
  function show(id) {
    if (id === currentId && panel.style.display !== "none") return;
    currentId = id;
    panel.style.display = "block";
    idLine.textContent = id;
    body.textContent = "Loading...";
    getRec(id).then(function (rec) { if (id === currentId) render(id, rec); });
  }

  /* ---------------------------------------------------------------
     Glacier view (inside the details popup)
  --------------------------------------------------------------- */
  function prepareShape(geom, term) {
    var polys = geom && geom.type === "Polygon" ? [geom.coordinates]
              : geom && geom.type === "MultiPolygon" ? geom.coordinates : null;
    if (!polys) return null;
    var minLon = Infinity, maxLon = -Infinity, minLat = Infinity, maxLat = -Infinity;
    polys.forEach(function (p) { p.forEach(function (ring) { ring.forEach(function (c) {
      if (c[0] < minLon) minLon = c[0];
      if (c[0] > maxLon) maxLon = c[0];
      if (c[1] < minLat) minLat = c[1];
      if (c[1] > maxLat) maxLat = c[1];
    }); }); });
    var lon0 = (minLon + maxLon) / 2, lat0 = (minLat + maxLat) / 2;
    var kx = 111320 * Math.cos(lat0 * Math.PI / 180), ky = 110540;
    function proj(c) { return [(c[0] - lon0) * kx, (c[1] - lat0) * ky]; }
    var rings = [];
    polys.forEach(function (p) { p.forEach(function (ring) { rings.push(ring.map(proj)); }); });
    var bx0 = (minLon - lon0) * kx, bx1 = (maxLon - lon0) * kx;
    var by0 = (minLat - lat0) * ky, by1 = (maxLat - lat0) * ky;

    function inside(x, y) {
      var c = false;
      for (var r = 0; r < rings.length; r++) {
        var ring = rings[r];
        for (var i = 0, j = ring.length - 1; i < ring.length; j = i++) {
          var xi = ring[i][0], yi = ring[i][1], xj = ring[j][0], yj = ring[j][1];
          if (((yi > y) !== (yj > y)) && (x < (xj - xi) * (y - yi) / (yj - yi) + xi)) c = !c;
        }
      }
      return c;
    }
    /* sample points inside the glacier; they tell us how much area lies beyond a given distance */
    var pts = [], step = Math.sqrt(Math.max(1, (bx1 - bx0) * (by1 - by0)) / 6000);
    for (var attempt = 0; attempt < 6; attempt++) {
      pts = [];
      for (var x = bx0 + step / 2; x < bx1; x += step) {
        for (var y = by0 + step / 2; y < by1; y += step) {
          if (inside(x, y)) pts.push([x, y]);
        }
      }
      if (pts.length >= 300 || step < 1) break;
      step *= 0.6;
    }
    if (!pts.length) return null;
    var cx = 0, cy = 0;
    pts.forEach(function (p) { cx += p[0]; cy += p[1]; });
    cx /= pts.length; cy /= pts.length;

    var T = null, dist = null;
    if (term && isFinite(term[0]) && isFinite(term[1])) {
      T = proj(term);
      dist = pts.map(function (p) { return Math.hypot(p[0] - T[0], p[1] - T[1]); }).sort(function (a, b) { return a - b; });
    }
    return { rings: rings, bx0: bx0, bx1: bx1, by0: by0, by1: by1, C: [cx, cy], T: T, dist: dist };
  }

  function buildViewer(root, rec, shape) {
    var scens = ALL.filter(function (sc) { return rec[sc]; });
    if (!scens.length) {
      root.appendChild(el("div", "gv-note", "No scenario data for this glacier."));
      return null;
    }
    if (scens.indexOf(viewerScen) < 0) viewerScen = scens[0];
    var y0 = rec.years[0], y1 = rec.years[1];
    var yearF = y0, playing = false, raf = null, lastTs = 0;

    var W = 380, H = 300, dpr = window.devicePixelRatio || 1;
    root.appendChild(el("div", "gv-title", "Glacier view"));
    var readout = el("div", "gv-readout");
    root.appendChild(readout);
    var canvas = document.createElement("canvas");
    canvas.className = "gv-canvas";
    canvas.width = Math.round(W * dpr);
    canvas.height = Math.round(H * dpr);
    canvas.style.width = W + "px";
    canvas.style.height = H + "px";
    root.appendChild(canvas);
    var ctx = canvas.getContext("2d");
    ctx.setTransform(dpr, 0, 0, dpr, 0, 0);

    var bar = el("div", "gv-bar");
    var playBtn = el("button", "gv-play", "\u25b6");
    playBtn.title = "Play";
    var slider = document.createElement("input");
    slider.type = "range";
    slider.className = "gv-slider";
    slider.min = y0;
    slider.max = y1;
    slider.step = 1;
    slider.value = y0;
    bar.appendChild(playBtn);
    bar.appendChild(slider);
    root.appendChild(bar);

    function makeSelect(label, options, value, onchange) {
      var wrap = el("label", "gv-opt");
      wrap.appendChild(el("span", "", label));
      var sel = document.createElement("select");
      options.forEach(function (o) {
        var op = document.createElement("option");
        op.value = o[0];
        op.textContent = o[1];
        sel.appendChild(op);
      });
      sel.value = value;
      sel.addEventListener("change", function () { onchange(sel.value); });
      wrap.appendChild(sel);
      return wrap;
    }
    var opts = el("div", "gv-opts");
    opts.appendChild(makeSelect("Scenario", scens.map(function (sc) { return [sc, LABELS[sc]]; }),
      viewerScen, function (v) { viewerScen = v; render(yearF); }));
    opts.appendChild(makeSelect("Model range", [["median", "Typical (median of models)"],
      ["high", "Smaller loss (90th percentile)"], ["low", "Larger loss (10th percentile)"]],
      viewerRange, function (v) { viewerRange = v; render(yearF); }));
    opts.appendChild(makeSelect("Shrink style", [["retreat", "Retreat from the lowest end"],
      ["uniform", "Shrink evenly toward the centre"]],
      viewerMode, function (v) { viewerMode = v; render(yearF); }));
    opts.appendChild(makeSelect("Speed", [["5", "Slow"], ["10", "Normal"], ["20", "Fast"]],
      String(viewerSpeed), function (v) { viewerSpeed = +v; }));
    root.appendChild(opts);
    root.appendChild(el("div", "gv-note",
      "Schematic: the model gives how much ice is left, not where. The ice is drawn retreating from the " +
      "lowest end (the terminus); the dashed line is the 2000 outline."));

    /* fit the outline to the canvas */
    var pad = 26;
    var sw = Math.max(1, shape.bx1 - shape.bx0), sh = Math.max(1, shape.by1 - shape.by0);
    var scale = Math.min((W - 2 * pad) / sw, (H - 2 * pad) / sh);
    var mx = (shape.bx0 + shape.bx1) / 2, my = (shape.by0 + shape.by1) / 2;
    function P(p) { return [W / 2 + (p[0] - mx) * scale, H / 2 - (p[1] - my) * scale]; }
    function tracePath() {
      ctx.beginPath();
      shape.rings.forEach(function (ring) {
        ring.forEach(function (p, i) {
          var q = P(p);
          if (i === 0) ctx.moveTo(q[0], q[1]); else ctx.lineTo(q[0], q[1]);
        });
        ctx.closePath();
      });
    }
    function radiusFor(f) {
      var n = shape.dist.length;
      if (f >= 1) return 0;
      var idx = Math.min(n - 1, Math.floor((1 - f) * n));
      return shape.dist[idx];
    }

    function fracAt(y) {
      var key = viewerRange === "high" ? "p90" : viewerRange === "low" ? "p10" : "median";
      var arr = rec[viewerScen][key];
      var f = y - y0, i = Math.floor(f), t = f - i;
      var a = arr[Math.max(0, Math.min(arr.length - 1, i))];
      var b = arr[Math.max(0, Math.min(arr.length - 1, i + 1))];
      if (a === null) a = b;
      if (b === null) b = a;
      if (a === null) return null;
      return (a + (b - a) * t) / 100;
    }

    function draw(frac) {
      ctx.clearRect(0, 0, W, H);
      ctx.fillStyle = "#eef3f7";
      ctx.fillRect(0, 0, W, H);

      tracePath();                                   /* ghost of the 2000 outline */
      ctx.fillStyle = "#dde3ea";
      ctx.fill("evenodd");
      ctx.setLineDash([5, 4]);
      ctx.strokeStyle = "#8794a3";
      ctx.lineWidth = 1.2;
      ctx.stroke();
      ctx.setLineDash([]);

      var shown = Math.min(1, frac);
      if (shown > 0.0005) {
        ctx.save();
        if (viewerMode === "retreat" && shape.T) {
          var t = P(shape.T), r = radiusFor(shown) * scale;
          ctx.beginPath();
          ctx.rect(0, 0, W, H);
          ctx.moveTo(t[0] + r, t[1]);
          ctx.arc(t[0], t[1], r, 0, 2 * Math.PI);
          ctx.clip("evenodd");
        } else {
          var k = Math.sqrt(shown), c = P(shape.C);
          ctx.translate(c[0], c[1]);
          ctx.scale(k, k);
          ctx.translate(-c[0], -c[1]);
        }
        tracePath();
        ctx.fillStyle = "#ffffff";
        ctx.fill("evenodd");
        ctx.fillStyle = "rgba(90, 169, 214, 0.55)";
        ctx.fill("evenodd");
        ctx.lineWidth = 1.6;
        ctx.strokeStyle = "#1f6f9f";
        ctx.stroke();
        ctx.restore();
      }
      if (shape.T) {                                 /* terminus position in 2000 */
        var tp = P(shape.T);
        ctx.beginPath();
        ctx.arc(tp[0], tp[1], 3.5, 0, 2 * Math.PI);
        ctx.fillStyle = "#f79420";
        ctx.fill();
        ctx.lineWidth = 1;
        ctx.strokeStyle = "#fff";
        ctx.stroke();
      }
      /* scale bar and north arrow */
      var nice = [10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000], meters = nice[0];
      nice.forEach(function (m) { if (m * scale <= W * 0.28) meters = m; });
      var len = meters * scale;
      ctx.strokeStyle = "#333";
      ctx.lineWidth = 2;
      ctx.beginPath();
      ctx.moveTo(12, H - 14);
      ctx.lineTo(12 + len, H - 14);
      ctx.stroke();
      ctx.fillStyle = "#333";
      ctx.font = "11px system-ui, Arial, sans-serif";
      ctx.textAlign = "left";
      ctx.fillText(meters >= 1000 ? (meters / 1000) + " km" : meters + " m", 12, H - 20);
      ctx.textAlign = "center";
      ctx.fillText("N", W - 16, 18);
      ctx.beginPath();
      ctx.moveTo(W - 16, 22);
      ctx.lineTo(W - 16, 38);
      ctx.stroke();
    }

    function render(y) {
      var f = fracAt(y);
      var yr = Math.round(y);
      slider.value = yr;
      selYear = yr;
      if (markerApi) markerApi(yr);
      draw(f === null ? 0 : f);
      readout.textContent = "";
      readout.appendChild(el("span", "gv-year", String(yr)));
      var txt;
      if (f === null) txt = "  no result";
      else if (f < 0.005) txt = "  gone (less than 1% of the 2000 area left)";
      else txt = "  " + km2(rec.area_km2 * f) + " km2 left, " + Math.round(f * 100) + "% of the 2000 area";
      readout.appendChild(document.createTextNode(txt));
    }

    function setPlayIcon() {
      playBtn.textContent = playing ? "\u275a\u275a" : "\u25b6";
      playBtn.title = playing ? "Pause" : "Play";
    }
    function tick(ts) {
      if (!playing) return;
      if (!lastTs) lastTs = ts;
      yearF += (ts - lastTs) / 1000 * viewerSpeed;
      lastTs = ts;
      if (yearF >= y1) { yearF = y1; playing = false; setPlayIcon(); }
      render(yearF);
      if (playing) raf = requestAnimationFrame(tick);
    }
    playBtn.addEventListener("click", function () {
      if (playing) { playing = false; setPlayIcon(); return; }
      if (yearF >= y1) yearF = y0;
      playing = true;
      lastTs = 0;
      setPlayIcon();
      raf = requestAnimationFrame(tick);
    });
    slider.addEventListener("input", function () {
      playing = false;
      setPlayIcon();
      yearF = +slider.value;
      render(yearF);
    });

    render(yearF);
    return {
      setYear: function (y) {
        playing = false;
        setPlayIcon();
        yearF = Math.max(y0, Math.min(y1, y));
        render(yearF);
      },
      destroy: function () {
        playing = false;
        if (raf) cancelAnimationFrame(raf);
      }
    };
  }

  function relayout(popup) {            /* do not call popup.update(): it would rebuild the content */
    try {
      if (popup._updateLayout) popup._updateLayout();
      if (popup._updatePosition) popup._updatePosition();
      if (popup._adjustPan) popup._adjustPan();
    } catch (err) {}
  }

  function geometryOf(popup) {
    var src = popup && popup._source;
    if (!src) return null;
    var g = src.feature && src.feature.geometry;
    if (!g && src.toGeoJSON) {
      try { g = src.toGeoJSON().geometry; } catch (err) {}
    }
    return g || null;
  }

  function mountViewer(popup, id) {
    var node = popup && popup._contentNode;
    if (!node || node.querySelector(".gv-root")) return;
    if (viewer) { viewer.destroy(); viewer = null; }
    var geom = geometryOf(popup);
    var root = el("div", "gv-root");
    root.appendChild(el("div", "gv-note", "Loading glacier view..."));
    node.appendChild(root);
    popup.options.maxWidth = 460;
    popup.options.minWidth = 410;
    relayout(popup);
    getRec(id).then(function (rec) {
      if (!root.parentNode) return;
      root.textContent = "";
      if (!rec) {
        root.appendChild(el("div", "gv-note", "No projection available for this glacier, so there is nothing to animate."));
        relayout(popup);
        return;
      }
      var shape = prepareShape(geom, rec.term);
      if (!shape) {
        root.appendChild(el("div", "gv-note", "The outline is not available for the glacier view."));
        relayout(popup);
        return;
      }
      viewer = buildViewer(root, rec, shape);
      relayout(popup);
    });
  }

  function closeViewer() {
    if (viewer) { viewer.destroy(); viewer = null; }
    selYear = null;
    if (markerApi) markerApi(null);
  }

  /* ---------------------------------------------------------------
     Wiring to the Leaflet map
  --------------------------------------------------------------- */
  function findId(layer) {
    var props = (layer && layer.feature && layer.feature.properties) || {};
    for (var k in props) {
      var v = props[k];
      if (typeof v === "string") { var m = v.match(ID_RE); if (m) return m[0]; }
    }
    return null;
  }

  function attach(group) {
    if (group.__projHooked) return;
    group.__projHooked = true;
    group.on("click", function (e) {
      var id = findId(e.layer || e.target);
      if (id) show(id);
    });
  }

  function hook(map) {
    map.on("popupopen", function (e) {
      var node = e.popup && e.popup._contentNode;
      var m = (node ? node.innerHTML : "").match(ID_RE);
      if (m) { show(m[0]); mountViewer(e.popup, m[0]); }
    });
    map.on("popupclose", closeViewer);
    map.eachLayer(function (l) { if (l instanceof L.GeoJSON) attach(l); });
    map.on("layeradd", function (e) { if (e.layer instanceof L.GeoJSON) attach(e.layer); });
  }

  var tries = 0;
  function findMap() {
    try {
      for (var k in window) {
        if (/^map_[0-9a-f]+$/.test(k) && window.L && window[k] instanceof L.Map) return window[k];
      }
    } catch (err) {}
    return null;
  }
  function init() {
    var map = findMap();
    if (!map) { if (tries++ < 100) setTimeout(init, 100); return; }
    hook(map);
  }
  init();
})();
'''

CSS = r'''.proj-panel {
  position: fixed; top: 56px; right: 12px; width: 620px;
  max-width: calc(100vw - 24px); max-height: calc(100vh - 72px); overflow-y: auto;
  z-index: 1200; background: #fff; color: #222; border-radius: 8px;
  box-shadow: 0 2px 14px rgba(0, 0, 0, 0.3);
  font: 14px/1.45 system-ui, -apple-system, "Segoe UI", Arial, sans-serif;
  padding: 14px 16px 16px;
}
.proj-panel.proj-wide { width: 1000px; }
.proj-btn {
  position: absolute; top: 8px; border: 0; background: none; cursor: pointer;
  font-size: 22px; line-height: 1; color: #666; padding: 2px 6px;
}
.proj-close { right: 8px; }
.proj-expand { right: 40px; }
.proj-btn:hover { color: #000; }
.proj-heading { font-weight: 600; font-size: 16px; padding-right: 80px; }
.proj-id { color: #777; font-size: 12px; margin: 2px 0 8px; }
.proj-toggle { display: block; margin: 0 0 4px; font-size: 13px; color: #444; cursor: pointer; }
.proj-chart { margin: 4px 0 8px; }
.proj-chartwrap { position: relative; }
.proj-chartwrap svg { display: block; }
.proj-tip {
  position: absolute; top: 8px; right: 18px; background: rgba(255, 255, 255, 0.96);
  border: 1px solid #ddd; border-radius: 4px; padding: 5px 8px; font-size: 12px;
  pointer-events: none;
}
.proj-tip-year { font-weight: 600; color: #333; }
.proj-row { display: flex; gap: 8px; align-items: flex-start; margin: 0 0 6px; cursor: pointer; }
.proj-row input { margin-top: 4px; flex: none; }
.proj-swatch { flex: none; width: 24px; height: 0; border-top: 3px solid; margin-top: 9px; }
.proj-swatch.proj-dashed { border-top-style: dashed; }
.proj-details { margin-top: 6px; }
.proj-details summary { cursor: pointer; color: #555; margin: 0 0 6px; }
.proj-warn { background: #fff4e0; border-left: 3px solid #f79420; padding: 6px 8px; margin: 8px 0; }
.proj-note { color: #666; font-size: 12px; margin: 10px 0 0; }

/* glacier view inside the details popup */
.gv-root { width: 380px; margin-top: 10px; padding-top: 8px; border-top: 1px solid #ddd; }
.gv-title { font-weight: 600; margin-bottom: 4px; }
.gv-readout { margin-bottom: 6px; font-size: 13px; }
.gv-year { font-size: 20px; font-weight: 700; }
.gv-canvas { display: block; border-radius: 6px; border: 1px solid #d5dde5; }
.gv-bar { display: flex; align-items: center; gap: 10px; margin: 8px 0 4px; }
.gv-play {
  flex: none; width: 36px; height: 30px; border: 1px solid #1f6f9f; background: #1f6f9f;
  color: #fff; border-radius: 6px; cursor: pointer; font-size: 13px;
}
.gv-slider { flex: 1; }
.gv-opts { display: grid; grid-template-columns: 1fr 1fr; gap: 6px 10px; margin-top: 6px; }
.gv-opt { display: flex; flex-direction: column; font-size: 11px; color: #555; }
.gv-opt select { font-size: 12px; padding: 2px; margin-top: 2px; }
.gv-note { color: #666; font-size: 11px; margin-top: 8px; }

@media (max-width: 700px) {
  .proj-panel, .proj-panel.proj-wide {
    top: auto; bottom: 0; right: 0; left: 0; width: auto; max-width: none;
    max-height: 65vh; border-radius: 12px 12px 0 0;
  }
  .proj-expand { display: none; }
  .gv-root { width: 300px; }
  .gv-canvas { width: 300px !important; }
}
'''

(DOCS / 'projection_panel.js').write_text(JS, encoding='utf-8')
(DOCS / 'projection_panel.css').write_text(CSS, encoding='utf-8')

index = DOCS / 'index.html'
html = index.read_text(encoding='utf-8')
if 'projection_panel.js' in html:
    print('index.html already links the panel files (they were refreshed).')
else:
    css_tag = '<link rel="stylesheet" href="projection_panel.css"/>\n'
    js_tag = '<script src="projection_panel.js"></script>\n'
    html = html.replace('</head>', css_tag + '</head>', 1) if '</head>' in html else css_tag + html
    cut = html.rfind('</html>')
    html = html[:cut] + js_tag + html[cut:] if cut != -1 else html + js_tag
    index.write_text(html, encoding='utf-8')
    print('Linked the panel files in', index)
print('Projection files available:', len(list((DOCS / 'data' / 'proj').glob('RGI*.json'))))

index.html already links the panel files (they were refreshed).
Projection files available: 3340


 CELL 11 - ADD THE TERMINUS POSITION TO EACH GLACIER
   Reads the terminus (lowest end) position of every glacier from
   your RGI 7 table and adds it to the glacier's file in
   docs/data/proj/. The glacier view uses it to retreat the ice
   from the lowest end first.



In [ ]:


import json
from pathlib import Path
import pandas as pd
from pyogrio import read_dataframe

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists()), Path.cwd())
OUT = ROOT / 'docs' / 'data' / 'proj'

df = read_dataframe(ROOT / 'data' / 'processed' / 'nepal_glaciers.gpkg',
                    read_geometry=False, columns=['rgi_id', 'termlon', 'termlat'])
terms = {r.rgi_id: [round(float(r.termlon), 5), round(float(r.termlat), 5)]
         for r in df.itertuples() if pd.notna(r.termlon) and pd.notna(r.termlat)}

done = missing = 0
for p in OUT.glob('RGI*.json'):
    rec = json.loads(p.read_text())
    t = terms.get(rec['id'])
    if t:
        rec['term'] = t
        p.write_text(json.dumps(rec, separators=(',', ':')))
        done += 1
    else:
        missing += 1
print(f'Added the terminus to {done:,} glacier files; {missing} have no terminus position')
print('Example:', next(iter(terms.items())))

Added the terminus to 3,340 glacier files; 0 have no terminus position
Example: ('RGI2000-v7.0-G-14-35809', [80.56239, 30.47736])
